In [1]:
# ===== COMPLETE RFM + BEHAVIORAL FEATURE ENGINEERING =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

print("=== COMPLETE RFM + BEHAVIORAL FEATURE ENGINEERING ===")

df = pd.read_csv('../data/raw/customer_shopping_data.csv')

df['invoice_date'] = pd.to_datetime(df['invoice_date'], format='%d/%m/%Y')
exchange_rate = 2.70
df['price_inr'] = df['price'] * exchange_rate

print(f"Currency conversion applied: {exchange_rate} INR per TL")

print("\nCalculating RFM metrics...")
latest_date = df['invoice_date'].max()
print(f"Latest date in dataset: {latest_date}")

rfm_data = df.groupby('customer_id').agg({
    'invoice_date': lambda x: (latest_date - x.max()).days,
    'invoice_no': 'count',
    'price_inr': 'sum'
}).reset_index()

rfm_data.columns = ['customer_id', 'recency', 'frequency', 'monetary_inr']

print(f"RFM Data Shape: {rfm_data.shape}")

customer_demographics = df[['customer_id', 'gender', 'age', 'shopping_mall']].drop_duplicates()
rfm_full = rfm_data.merge(customer_demographics, on='customer_id', how='left')

print(f"RFM with demographics shape: {rfm_full.shape}")

print("\n=== ADDING BEHAVIORAL FEATURES ===")

print("1. Adding category preferences...")
category_behavior = df.groupby('customer_id')['category'].agg([
    ('favorite_category', lambda x: x.mode()[0] if len(x.mode()) > 0 else 'Unknown'),
    ('category_variety', 'nunique'),
    ('is_fashion_shopper', lambda x: 1 if any(cat in ['Clothing', 'Shoes', 'Cosmetics'] for cat in x) else 0)
]).reset_index()
 
print("2. Adding payment behavior...")
payment_behavior = df.groupby('customer_id')['payment_method'].agg([
    ('preferred_payment', lambda x: x.mode()[0]),
    ('payment_flexibility', 'nunique')
]).reset_index()

print("3. Adding cart behavior...")
cart_behavior = df.groupby('customer_id').agg({
    'quantity': 'mean'
}).reset_index()
cart_behavior.columns = ['customer_id', 'avg_items_per_order']

max_items = df.groupby('customer_id')['quantity'].max().reset_index()
max_items.columns = ['customer_id', 'max_items_per_order']

avg_order_value = df.groupby('customer_id')['price_inr'].mean().reset_index()
avg_order_value.columns = ['customer_id', 'avg_order_value']

cart_behavior = cart_behavior.merge(max_items, on='customer_id')
cart_behavior = cart_behavior.merge(avg_order_value, on='customer_id')

print("4. Adding time behavior...")
df['purchase_hour'] = df['invoice_date'].dt.hour
df['day_of_week'] = df['invoice_date'].dt.dayofweek
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

time_behavior = df.groupby('customer_id').agg({
    'purchase_hour': 'mean',
    'is_weekend': 'mean',
}).reset_index()
time_behavior.columns = ['customer_id', 'preferred_hour', 'weekend_ratio']

print("\nMerging all features...")
rfm_with_behavior = rfm_full.merge(category_behavior, on='customer_id', how='left')
rfm_with_behavior = rfm_with_behavior.merge(payment_behavior, on='customer_id', how='left')
rfm_with_behavior = rfm_with_behavior.merge(cart_behavior, on='customer_id', how='left')
rfm_with_behavior = rfm_with_behavior.merge(time_behavior, on='customer_id', how='left')

print(f"Final dataset shape: {rfm_with_behavior.shape}")

print("\n FINAL FEATURE SET:")
features = [
    'recency', 'frequency', 'monetary_inr',                    
    'category_variety', 'is_fashion_shopper',                  
    'payment_flexibility',                                     
    'avg_items_per_order', 'max_items_per_order', 'avg_order_value',  
    'preferred_hour', 'weekend_ratio'                          
]

print("RFM + Behavioral features:")
for feature in features:
    if feature in rfm_with_behavior.columns:
        print(f"   {feature}")

rfm_full = rfm_with_behavior.copy()

rfm_full.to_csv('../data/processed/rfm_data_with_behavior.csv', index=False)
print(f"\n Data saved to '../data/processed/rfm_data_with_behavior.csv'")

print("\n RFM + BEHAVIORAL FEATURES COMPLETED!")
print(f"Ready for modeling with {len(features)} features")

=== COMPLETE RFM + BEHAVIORAL FEATURE ENGINEERING ===
Currency conversion applied: 2.7 INR per TL

Calculating RFM metrics...
Latest date in dataset: 2023-03-08 00:00:00
RFM Data Shape: (99457, 4)
RFM with demographics shape: (99457, 7)

=== ADDING BEHAVIORAL FEATURES ===
1. Adding category preferences...
2. Adding payment behavior...
3. Adding cart behavior...
4. Adding time behavior...

Merging all features...
Final dataset shape: (99457, 17)

 FINAL FEATURE SET:
RFM + Behavioral features:
   recency
   frequency
   monetary_inr
   category_variety
   is_fashion_shopper
   payment_flexibility
   avg_items_per_order
   max_items_per_order
   avg_order_value
   preferred_hour
   weekend_ratio

 Data saved to '../data/processed/rfm_data_with_behavior.csv'

 RFM + BEHAVIORAL FEATURES COMPLETED!
Ready for modeling with 11 features
